# Descarga de los datasets de phmd a Drive

Este notebook descarga **todos los datasets de series temporales** que ofrece la librería `phmd` y los guarda en `MyDrive/fm_fl_phmd/raw/`. Es el paso previo a cualquier análisis o entrenamiento.

Ideas clave del diseño:

- **Drive como almacén único.** Los `.zip` originales se quedan en Drive, no en el disco local de la VM (que es efímero). De esta forma, descargamos una sola vez aunque cambiemos de máquina.
- **Sin descomprimir.** Mantenemos los archivos comprimidos (`unzip=False`). Descomprimir crearía miles de ficheros pequeños y Drive es lento con eso. Se descomprimen sobre la marcha cuando un notebook concreto los necesite.
- **Reentrante.** Se mantiene un fichero `_manifest.json` con el estado de cada dataset (descargado / fallido / pendiente). Si la sesión de Colab se corta, basta con volver a lanzar el notebook y continúa donde se quedó.
- **Idempotente.** Si un dataset ya está en Drive, no se vuelve a descargar.

Volumen total esperado: alrededor de **50–60 GB** distribuidos entre 55 datasets. El más grande (NCMAPSS) es de unos 14 GB; el más pequeño, de pocos KB.

## Setup

Dos líneas: montamos Drive y lanzamos `colab_init.sh`, que se encarga del resto (clave SSH, clone del repo, instalación de dependencias). Ese script lo dejó en Drive el `setup/colab_bootstrap.ipynb` la primera vez.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

## 1. Rutas de trabajo

Definimos dónde vive cada cosa en Drive. Todas las rutas penden de `MyDrive/fm_fl_phmd/`:

- `raw/` — carpeta raíz para los datasets crudos.
- `raw/datasets/` — subcarpeta donde `phmd` deposita los `.zip` (esto lo decide la propia librería).
- `raw/_manifest.json` — registro contable de qué se ha descargado, con qué tamaño y cuándo.
- `logs/download_datasets.log` — log de esta ejecución por si hay que revisar errores.

In [ ]:
from pathlib import Path

# Raíz del proyecto en Drive (montado en la VM bajo /content/drive)
DRIVE_ROOT = Path('/content/drive/MyDrive/fm_fl_phmd')

# Carpeta donde phmd guardará los .zip y el manifest
RAW_DIR  = DRIVE_ROOT / 'raw'
MANIFEST = RAW_DIR / '_manifest.json'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Carpeta de logs
LOG_DIR  = DRIVE_ROOT / 'logs'
LOG_FILE = LOG_DIR / 'download_datasets.log'
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Datos en : {RAW_DIR}')
print(f'Manifest : {MANIFEST}')
print(f'Log      : {LOG_FILE}')

## 2. Catálogo y plan de descarga

Le preguntamos a `phmd` qué datasets de tipo *time-series* ofrece y comparamos esa lista con el `_manifest.json` para decidir, dataset por dataset, si toca descargarlo o si ya está hecho.

Posibles estados de cada dataset en el manifest:

- `ok` — descargado correctamente.
- `failed` — falló (típicamente por un bug en la metadata de phmd para ese dataset).
- `manual_download` — phmd marca este dataset como descarga manual.
- (sin entrada) — todavía no se ha intentado.

In [ ]:
import json
from phmd import datasets as phmd_datasets

# Lista completa de datasets time-series según phmd
TS_NAMES = sorted(
    phmd_datasets.Dataset.search(nature='time-series', return_names=True)
)
print(f'Total datasets time-series en phmd: {len(TS_NAMES)}')

# Cargamos el manifest si ya existe; si no, lo inicializamos vacío
if MANIFEST.exists():
    manifest = json.loads(MANIFEST.read_text())
else:
    manifest = {'datasets': {}}

# phmd deposita los ficheros dentro de raw/datasets/ (lo decide la propia librería)
DATASETS_SUBDIR = RAW_DIR / 'datasets'


def archivos_dataset(nombre):
    """Devuelve los ficheros que pertenecen al dataset 'nombre'.

    Hay que tener en cuenta que phmd a veces escribe el nombre con minúsculas
    (curves.zip), a veces lo divide (CMAPSS_train.zip + CMAPSS_test.zip),
    a veces sin extensión. Por eso comparamos en minúsculas y permitimos
    prefijos.
    """
    if not DATASETS_SUBDIR.exists():
        return []
    nombre_lc = nombre.lower()
    return [
        p for p in DATASETS_SUBDIR.iterdir()
        if p.is_file() and (
            p.stem.lower() == nombre_lc
            or p.stem.lower().startswith(nombre_lc + '_')
            or p.name.lower().startswith(nombre_lc + '.')
        )
    ]


# Repartimos los datasets entre tres listas según su estado actual
plan = {'pendientes': [], 'hechos': [], 'reintentar': []}
for nombre in TS_NAMES:
    entrada = manifest['datasets'].get(nombre, {})
    estado  = entrada.get('status')
    en_disco = bool(archivos_dataset(nombre))

    if estado == 'ok' and en_disco:
        plan['hechos'].append(nombre)
    elif estado == 'failed':
        plan['reintentar'].append(nombre)
        plan['pendientes'].append(nombre)
    elif en_disco and estado != 'manual_download':
        # Los ficheros están pero el manifest no lo dice (manifest desfasado)
        plan['hechos'].append(nombre)
    else:
        plan['pendientes'].append(nombre)

print(f"Ya descargados   : {len(plan['hechos'])}")
print(f"Por descargar    : {len(plan['pendientes'])}")
print(f"Reintento previo : {len(plan['reintentar'])}")
if plan['pendientes']:
    print('Primeros 10 a descargar:', plan['pendientes'][:10])

## 3. Bucle de descarga

Vamos uno a uno por los datasets pendientes. Para cada uno:

1. Instanciamos `Dataset(nombre, cache_dir=RAW_DIR)` para que phmd sepa dónde guardar.
2. Si en su metadata indica `manual_download=True`, lo saltamos.
3. Llamamos a `ds.download(unzip=False)`.
4. Anotamos el resultado en el manifest (tamaño, lista de ficheros, tiempo, timestamp).
5. Si algo lanza una excepción, la registramos y seguimos con el siguiente.

Para descargas muy largas (NCMAPSS o FCLB19 pueden tardar varios minutos cada uno), conviene activar **Runtime → Manage sessions → Background execution** para que la descarga continúe aunque cerremos el navegador.

In [ ]:
import json
import time
import traceback
from datetime import datetime, timezone

from phmd import datasets as phmd_datasets


def tam_bytes(nombre):
    """Tamaño total en bytes de los ficheros del dataset en Drive."""
    return sum(f.stat().st_size for f in archivos_dataset(nombre))


def guardar_manifest():
    """Escritura atómica del manifest (primero a un .tmp y luego rename).

    Si la sesión se corta justo durante la escritura, esto evita corromper el
    fichero porque el rename es atómico en POSIX.
    """
    tmp = MANIFEST.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(manifest, indent=2, sort_keys=True))
    tmp.replace(MANIFEST)


def log(msg):
    """Imprime y guarda en el log de Drive con timestamp."""
    linea = f'[{datetime.now(timezone.utc).isoformat(timespec="seconds")}] {msg}'
    print(linea)
    with LOG_FILE.open('a') as f:
        f.write(linea + '\n')


manifest.setdefault('datasets', {})
pendientes = plan['pendientes']
log(f'Inicio del bucle. {len(pendientes)} dataset(s) a descargar. cache_dir={RAW_DIR}')

for i, nombre in enumerate(pendientes, 1):
    log(f'[{i}/{len(pendientes)}] {nombre} ...')
    t0 = time.time()
    try:
        # phmd guarda en cache_dir/datasets/<NAME>.zip
        ds = phmd_datasets.Dataset(nombre, cache_dir=str(RAW_DIR))

        # Algunos datasets están marcados como descarga manual
        # (porque la fuente original no permite acceso programático)
        meta = getattr(ds, 'metadata', None) or getattr(ds, 'meta', {}) or {}
        if isinstance(meta, dict) and meta.get('manual_download'):
            manifest['datasets'][nombre] = {
                'status': 'manual_download',
                'note':   'descarga manual requerida',
                'ts':     datetime.now(timezone.utc).isoformat(timespec='seconds'),
            }
            guardar_manifest()
            log('  -> descarga manual: omitido')
            continue

        # Descarga propiamente dicha. unzip=False = nos quedamos con el .zip
        ds.download(force=False, unzip=False)

        size = tam_bytes(nombre)
        ficheros = [f.name for f in archivos_dataset(nombre)]
        manifest['datasets'][nombre] = {
            'status':     'ok',
            'size_bytes': size,
            'size_mb':    round(size / 1024 / 1024, 2),
            'files':      ficheros,
            'elapsed_s':  round(time.time() - t0, 1),
            'ts':         datetime.now(timezone.utc).isoformat(timespec='seconds'),
        }
        guardar_manifest()
        log(f'  -> ok ({manifest["datasets"][nombre]["size_mb"]} MB, '
            f'{manifest["datasets"][nombre]["elapsed_s"]} s)')

    except Exception as e:
        # Cualquier fallo lo registramos pero no abortamos el bucle
        err = f'{type(e).__name__}: {e}'
        manifest['datasets'][nombre] = {
            'status':    'failed',
            'error':     err,
            'traceback': traceback.format_exc(),
            'ts':        datetime.now(timezone.utc).isoformat(timespec='seconds'),
        }
        guardar_manifest()
        log(f'  -> FALLO: {err}')

log('Bucle terminado.')

## 4. Resumen final

Recorremos el manifest, recalculamos tamaños mirando los ficheros reales en Drive (por si quedó algo desfasado de iteraciones anteriores) y mostramos un balance: cuántos datasets en cada estado, GB totales y lista de fallos.

In [ ]:
import json
from collections import Counter

manifest = json.loads(MANIFEST.read_text())

# Recalculamos tamaños desde el sistema de ficheros, no fiándonos del manifest
for nombre, entrada in manifest['datasets'].items():
    if entrada.get('status') != 'ok':
        continue
    if not DATASETS_SUBDIR.exists():
        continue
    ficheros = archivos_dataset(nombre)
    size = sum(f.stat().st_size for f in ficheros)
    entrada['size_bytes'] = size
    entrada['size_mb']    = round(size / 1024 / 1024, 2)
    entrada['files']      = [f.name for f in ficheros]

# Guardamos el manifest reparado
tmp = MANIFEST.with_suffix('.json.tmp')
tmp.write_text(json.dumps(manifest, indent=2, sort_keys=True))
tmp.replace(MANIFEST)

# Estadísticas globales
estados   = Counter(d.get('status', 'unknown') for d in manifest['datasets'].values())
total_mb  = sum(d.get('size_mb', 0) for d in manifest['datasets'].values())

print(f'Estados      : {dict(estados)}')
print(f'Tamaño total : {total_mb:,.0f} MB ({total_mb/1024:.2f} GB)')
print()

# Detalle de los que han fallado
fallos = [(n, d) for n, d in manifest['datasets'].items() if d.get('status') == 'failed']
if fallos:
    print('Datasets que han fallado:')
    for n, d in fallos:
        print(f'  - {n:30s} {d.get("error", "")[:120]}')

# Detalle de los marcados como descarga manual
manuales = [n for n, d in manifest['datasets'].items() if d.get('status') == 'manual_download']
if manuales:
    print()
    print('Datasets de descarga manual:')
    for n in manuales:
        print(f'  - {n}')

## 5. Top 10 por tamaño

Un vistazo a los diez datasets más pesados en Drive. Útil para saber a qué hay que prestar atención: los grandes son los que más tiempo van a comer al descomprimir o procesar.

In [ ]:
ok = [
    (n, d) for n, d in manifest['datasets'].items()
    if d.get('status') == 'ok'
]
ok.sort(key=lambda x: x[1].get('size_mb', 0), reverse=True)

for n, d in ok[:10]:
    print(f'  {n:30s} {d["size_mb"]:>10.1f} MB')